# LexAI — Notebook 04: AI Argument Generation

Demonstrates the dual-sided AI legal argument engine:
- **Prosecution arguments**: opening, main arguments, closing, evidence submissions
- **Defense arguments**: counter-arguments, acquittal grounds, cross-examination questions
- **Argument strength comparison** with radar chart visualisation
- **Cross-examination generator** (15+ questions per witness)

Note: Without GROQ_API_KEY set, responses use mock data for demonstration.

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
print('LexAI Argument Generation Demo')

## 1. Load Sample Case Data

In [ ]:
import json, pathlib

case_path = pathlib.Path('../data/cases/sample_cases/case_002/case_metadata.json')
if case_path.exists():
    with open(case_path) as f:
        case_data = json.load(f)
else:
    # Fallback mock case
    case_data = {
        'metadata': {
            'case_id': 'CASE-2024-CRM-002',
            'case_type': 'Criminal',
            'charges': ['IPC 420 — Cheating', 'IPC 468 — Forgery for purpose of cheating'],
            'court': 'Sessions Court, Mumbai',
            'status': 'Trial',
            'plaintiff_name': 'State of Maharashtra',
            'defendant_name': 'Rahul Mehta',
        }
    }

print('Case ID:', case_data['metadata']['case_id'])
print('Type:', case_data['metadata']['case_type'])
print('Charges:', case_data['metadata']['charges'])
print('Court:', case_data['metadata']['court'])

## 2. Initialize Argument Generator

In [ ]:
from src.argument_generator import LegalArgumentGenerator

generator = LegalArgumentGenerator()
print('Generator initialized')
print('Available styles:', list(generator.ARGUMENT_STYLES.keys()))
for style, desc in generator.ARGUMENT_STYLES.items():
    print(f'  {style}: {desc}')

## 3. Sample Evidence Analyses

In [ ]:
evidence_analyses = [
    {
        'file_name': 'fir_report.pdf',
        'doc_type': 'pdf',
        'evidence_strength': 8.5,
        'summary': 'FIR filed at Andheri Police Station reporting Rs.45L fraud via fake investment scheme',
        'entities': {'persons': ['Rahul Mehta', 'Priya Kapoor'], 'amounts': ['Rs. 45,00,000']}
    },
    {
        'file_name': 'bank_statements.pdf',
        'doc_type': 'pdf',
        'evidence_strength': 9.0,
        'summary': 'Multiple bank transfers from victim accounts to accused shell companies totaling Rs.45L',
        'entities': {'organisations': ['Tech Ventures Pvt Ltd', 'Alpha Investments'], 'amounts': ['Rs. 45,00,000']}
    },
    {
        'file_name': 'contract_document.pdf',
        'doc_type': 'pdf',
        'evidence_strength': 7.5,
        'summary': 'Investment agreement with forged signatures and false guarantees',
        'entities': {'persons': ['Rahul Mehta'], 'legal_charges': ['420', '468']}
    },
]

print(f'Evidence items: {len(evidence_analyses)}')
avg_strength = sum(e['evidence_strength'] for e in evidence_analyses) / len(evidence_analyses)
print(f'Average evidence strength: {avg_strength:.1f}/10')

## 4. Generate Prosecution Argument

In [ ]:
prosecution = generator.generate_prosecution_argument(
    case_data=case_data,
    evidence_analyses=evidence_analyses,
    legal_context={},
    precedents=[],
    style='balanced'
)

print('=== PROSECUTION ARGUMENT ===')
print(f'Side: {prosecution["side"]}')
print(f'Strength Score: {prosecution["argument_strength_score"]}/10')
print(f'\nOpening Statement (excerpt):')
print(prosecution['opening_statement'][:400])
print('...')

In [ ]:
print('Key Arguments:')
for i, arg in enumerate(prosecution.get('key_arguments', [])[:3], 1):
    print(f'  {i}. {arg}')

print('\nEvidence Submissions:')
for es in prosecution.get('evidence_submissions', [])[:3]:
    print(f'  - {es}')

## 5. Generate Defense Argument

In [ ]:
defense = generator.generate_defense_argument(
    case_data=case_data,
    evidence_analyses=evidence_analyses,
    legal_context={},
    precedents=[],
    style='balanced'
)

print('=== DEFENSE ARGUMENT ===')
print(f'Side: {defense["side"]}')
print(f'Strength Score: {defense["argument_strength_score"]}/10')
print(f'\nOpening Statement (excerpt):')
print(defense['opening_statement'][:400])
print('...')

print('\nAcquittal Grounds:')
for i, ground in enumerate(defense.get('acquittal_grounds', [])[:3], 1):
    print(f'  {i}. {ground}')

## 6. Argument Strength Comparison

In [ ]:
comparison = generator.compare_argument_strength(prosecution, defense)

print('=== STRENGTH COMPARISON ===')
print(f'Winner Prediction: {comparison["winner_prediction"]}')
print(f'Prosecution Overall: {comparison["overall_prosecution_score"]:.1f}/10')
print(f'Defense Overall: {comparison["overall_defense_score"]:.1f}/10')
print(f'\nKey Advantage: {comparison.get("key_advantage", "")}') 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = ['Opening\nStrength', 'Evidence\nWeight', 'Legal\nCitations', 'Witness\nCredibility', 'Closing\nImpact']
N = len(categories)

# Simulated radar scores
prosecution_scores = [8.0, 9.0, 7.5, 7.0, 8.5]
defense_scores     = [7.5, 6.0, 8.0, 7.5, 7.0]

angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
prosecution_scores += prosecution_scores[:1]
defense_scores     += defense_scores[:1]
angles             += angles[:1]

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))
ax.plot(angles, prosecution_scores, 'o-', linewidth=2, color='#c0392b', label='Prosecution')
ax.fill(angles, prosecution_scores, alpha=0.15, color='#c0392b')
ax.plot(angles, defense_scores, 'o-', linewidth=2, color='#27ae60', label='Defense')
ax.fill(angles, defense_scores, alpha=0.15, color='#27ae60')
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=10)
ax.set_ylim(0, 10)
ax.set_title('Prosecution vs Defense — Argument Strength Radar', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

## 7. Cross-Examination Generator

In [ ]:
witness = {
    'name': 'Priya Kapoor',
    'role': 'victim/complainant',
    'statement': 'I invested Rs.45 lakhs in the accused scheme after seeing promotional materials guaranteeing 30% returns within 6 months.'
}

cross_exam = generator.generate_cross_examination(
    witness_info=witness,
    case_data=case_data,
    side='defense'
)

print(f'Cross-examination questions for {witness["name"]}:')
print(f'Generated {len(cross_exam.get("questions", []))} questions\n')
for i, q in enumerate(cross_exam.get('questions', [])[:8], 1):
    print(f'{i:2d}. {q}')

## Summary

The argument generator produces:
- **Full prosecution case**: opening → evidence → key arguments → closing
- **Full defense case**: counter-arguments → acquittal grounds → closing
- **Radar chart data** for strength comparison across 5 dimensions
- **Cross-examination questions** tailored to witness role and case type
- **3 styles**: Aggressive / Conservative / Balanced

Next: See Notebook 05 for ML-based verdict prediction.